In [0]:
# Databricks notebook source
# =============================================================================
# DEVICE LINKAGE RISK MART — FINAL GOLD MART
# =============================================================================
# grain: one row per device
# builds device-level fraud features by aggregating from the base enriched
# transactions table, then propagates customer and merchant risk from
# the other three gold marts.
# =============================================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

catalog = "ubuntu_bank_fraud"
gold = f"{catalog}.gold"

batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# load all four source tables — confirmed they exist and have correct columns
base   = spark.table(f"{gold}.gold_base_enriched_transactions")
fiu    = spark.table(f"{gold}.gold_fiu_customer_velocity_network")
cube   = spark.table(f"{gold}.gold_customer_risk_profile_cube")
merch  = spark.table(f"{gold}.gold_merchant_syndicate_anomaly_mart")

print(f"base:   {base.count():,} rows, {len(base.columns)} cols")
print(f"fiu:    {fiu.count():,} rows, {len(fiu.columns)} cols")
print(f"cube:   {cube.count():,} rows, {len(cube.columns)} cols")
print(f"merch:  {merch.count():,} rows, {len(merch.columns)} cols")

base:   2,200,000 rows, 88 cols
fiu:    28,311 rows, 50 cols
cube:   28,311 rows, 56 cols
merch:  1,986 rows, 47 cols


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# aggregate every transaction up to device grain
# ---------------------------------------------------------------------------

device_agg = base.groupBy("device_id").agg(

    # ---- identity ----
    first("device_type", ignorenulls=True).alias("device_type"),
    first("dev_os", ignorenulls=True).alias("device_os"),
    first("dev_mfr", ignorenulls=True).alias("device_manufacturer"),
    first("device_model", ignorenulls=True).alias("device_model"),
    first("dev_risk_score", ignorenulls=True).alias("operational_risk_score"),
    max("is_primary_device").cast("int").alias("is_primary_device"),
    max("is_first_txn_on_device").cast("int").alias("has_first_txn"),
    first("dev_link_type", ignorenulls=True).alias("device_link_type"),

    # ---- network ----
    countDistinct("customer_id").alias("distinct_customers"),
    countDistinct("source_account_id").alias("distinct_accounts"),
    countDistinct("card_id").alias("distinct_cards"),
    countDistinct("merchant_id").alias("distinct_merchants"),

    # ---- transaction behaviour ----
    count("transaction_id").alias("total_transactions"),
    round(sum("transaction_amount"), 2).alias("total_amount"),
    round(avg("transaction_amount"), 2).alias("avg_transaction_amount"),
    max("transaction_amount").alias("max_transaction_amount"),
    countDistinct("transaction_date").alias("active_days"),
    min("transaction_date").alias("first_seen"),
    max("transaction_date").alias("last_seen"),

    # ---- time patterns ----
    count(when(hour("transaction_date").between(22, 23) | 
               hour("transaction_date").between(0, 5), lit(1))).alias("night_transactions"),
    count(when(dayofweek("transaction_date").isin(1, 7), lit(1))).alias("weekend_transactions"),

    # ---- channel patterns ----
    count(when(col("transaction_type").isin("RTC_Transfer", "EFT_Transfer"), 
               lit(1))).alias("rtc_eft_transactions"),

    # ---- silver features ----
    max("device_customer_count").alias("max_device_customer_count"),
    max("device_account_count").alias("max_device_account_count"),
    max("shared_device_flag").alias("shared_device_flag")
)

# ---- derived ratios ----
device_agg = device_agg \
    .withColumn("night_pct",
                when(col("total_transactions") > 0,
                     round(col("night_transactions") / col("total_transactions"), 3))
                .otherwise(0.0)) \
    .withColumn("weekend_pct",
                when(col("total_transactions") > 0,
                     round(col("weekend_transactions") / col("total_transactions"), 3))
                .otherwise(0.0)) \
    .withColumn("rtc_eft_pct",
                when(col("total_transactions") > 0,
                     round(col("rtc_eft_transactions") / col("total_transactions"), 3))
                .otherwise(0.0)) \
    .withColumn("avg_daily_transactions",
                when(col("active_days") > 0,
                     round(col("total_transactions") / col("active_days"), 1))
                .otherwise(0.0))

total_devices = device_agg.count()
print(f"unique devices: {total_devices:,}")

unique devices: 18,854


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# for each device, find all linked customers and pull their risk scores
# from the fiu mart and the enterprise risk cube
# then aggregate back to device level
# ---------------------------------------------------------------------------

# get unique device-customer pairs from the base table
device_customers = base.select("device_id", "customer_id").distinct()

# join to fiu mart — bring in fiu risk scores per customer
device_fiu = device_customers.alias("dc") \
    .join(fiu.select("customer_id", "fiu_risk_score", "fiu_risk_band",
                     "pass_through_flag", "rtc_abuse_flag").alias("f"),
          "customer_id", "left")

# join to enterprise risk cube — bring in enterprise risk scores
device_cube = device_customers.alias("dc") \
    .join(cube.select("customer_id", "enterprise_risk_score",
                      "enterprise_risk_band").alias("c"),
          "customer_id", "left")

# aggregate fiu risk to device level
device_fiu_agg = device_fiu.groupBy("device_id").agg(
    count(when(col("fiu_risk_band").isin("HIGH", "CRITICAL"), lit(1))).alias("linked_fiu_high_risk"),
    count(when(col("fiu_risk_band") == "CRITICAL", lit(1))).alias("linked_fiu_critical"),
    count(when(col("pass_through_flag") == 1, lit(1))).alias("linked_pass_through"),
    count(when(col("rtc_abuse_flag") == 1, lit(1))).alias("linked_rtc_abuse"),
    round(avg("fiu_risk_score"), 1).alias("avg_fiu_score"),
    max("fiu_risk_score").alias("max_fiu_score")
)

# aggregate enterprise risk to device level
device_cube_agg = device_cube.groupBy("device_id").agg(
    count(when(col("enterprise_risk_band").isin("HIGH", "CRITICAL"), lit(1))).alias("linked_ent_high_risk"),
    count(when(col("enterprise_risk_band") == "CRITICAL", lit(1))).alias("linked_ent_critical"),
    round(avg("enterprise_risk_score"), 1).alias("avg_ent_score"),
    max("enterprise_risk_score").alias("max_ent_score")
)

print(f"devices with fiu risk data: {device_fiu_agg.count():,}")
print(f"devices with enterprise risk data: {device_cube_agg.count():,}")

devices with fiu risk data: 18,854
devices with enterprise risk data: 18,854


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# for each device, find all merchants transacted from that device
# and pull their risk scores from the merchant syndicate mart
# ---------------------------------------------------------------------------

# get unique device-merchant pairs (only where merchant exists)
device_merchants = base.select("device_id", "merchant_id") \
    .filter(col("merchant_id").isNotNull()) \
    .distinct()

# join to merchant mart
device_merch_risk = device_merchants.alias("dm") \
    .join(merch.select("merchant_id", "merchant_risk_score",
                       "merchant_risk_band").alias("m"),
          "merchant_id", "left")

# aggregate to device level
device_merch_agg = device_merch_risk.groupBy("device_id").agg(
    count(when(col("merchant_risk_band").isin("HIGH", "CRITICAL"), lit(1))).alias("linked_high_risk_merchants"),
    count(when(col("merchant_risk_band") == "CRITICAL", lit(1))).alias("linked_critical_merchants"),
    round(avg("merchant_risk_score"), 1).alias("avg_merchant_risk_score"),
    max("merchant_risk_score").alias("max_merchant_risk_score")
)

print(f"devices with merchant risk data: {device_merch_agg.count():,}")

devices with merchant risk data: 18,854


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# join all the pieces together into one device-level dataframe
# start with the core device aggregation, then layer on risk propagation
# ---------------------------------------------------------------------------

device_profile = device_agg \
    .join(device_fiu_agg, "device_id", "left") \
    .join(device_cube_agg, "device_id", "left") \
    .join(device_merch_agg, "device_id", "left")

# fill nulls — devices with no linked customers or merchants get zeros
device_profile = device_profile.fillna(0, [
    "linked_fiu_high_risk", "linked_fiu_critical",
    "linked_pass_through", "linked_rtc_abuse",
    "linked_ent_high_risk", "linked_ent_critical",
    "linked_high_risk_merchants", "linked_critical_merchants"
]).fillna(0.0, [
    "avg_fiu_score", "avg_ent_score", "avg_merchant_risk_score"
])

print(f"device profile rows: {device_profile.count():,}")

device profile rows: 18,854


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# BEFORE scoring, understand the natural distribution of key metrics
# this tells us what "normal" looks like so we can calibrate thresholds
# ---------------------------------------------------------------------------

print("=" * 60)
print("POPULATION PROFILING — BEFORE SCORING")
print("=" * 60)

# distribution of customers per device
print("\ndistinct_customers per device:")
device_profile.select("distinct_customers").describe().show()
print("\ndistribution:")
device_profile.groupBy("distinct_customers").count() \
    .orderBy("distinct_customers").show(15)

# distribution of accounts per device
print("\ndistinct_accounts per device:")
device_profile.select("distinct_accounts").describe().show()

# linked high risk customers
print("\nlinked_fiu_high_risk distribution:")
device_profile.groupBy("linked_fiu_high_risk").count() \
    .orderBy("linked_fiu_high_risk").show(10)

# linked critical customers
print("\nlinked_fiu_critical distribution:")
device_profile.groupBy("linked_fiu_critical").count() \
    .orderBy("linked_fiu_critical").show(10)

POPULATION PROFILING — BEFORE SCORING

distinct_customers per device:
+-------+------------------+
|summary|distinct_customers|
+-------+------------------+
|  count|             18854|
|   mean| 1.736660655563806|
| stddev|1.1583738410821023|
|    min|                 1|
|    max|                20|
+-------+------------------+


distribution:
+------------------+-----+
|distinct_customers|count|
+------------------+-----+
|                 1| 9964|
|                 2| 5668|
|                 3| 2227|
|                 4|  705|
|                 5|  202|
|                 6|   48|
|                 7|    7|
|                 8|    3|
|                17|    2|
|                18|    4|
|                19|   15|
|                20|    9|
+------------------+-----+


distinct_accounts per device:
+-------+------------------+
|summary| distinct_accounts|
+-------+------------------+
|  count|             18854|
|   mean|2.5441816060252465|
| stddev| 1.967225985731187|
|    min|      

In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# DEVICE RISK SCORE (0-100)
# ---------------------------------------------------------------------------
# a device at the median scores 0 for that component
# a device at the 95th percentile scores full points
# ---------------------------------------------------------------------------

metrics_for_scoring = [
    "distinct_customers", "distinct_accounts", "distinct_cards",
    "linked_fiu_high_risk", "linked_fiu_critical",
    "linked_ent_high_risk", "linked_ent_critical",
    "linked_high_risk_merchants", "linked_critical_merchants",
    "rtc_eft_pct", "night_pct"
]

medians = {}
p95s = {}

print("population benchmarks for scoring:")
print(f"  {'metric':<35} {'median':>8} {'p95':>8}")
print(f"  {'-'*35} {'-'*8} {'-'*8}")

for m in metrics_for_scoring:
    q = device_profile.approxQuantile(m, [0.5, 0.95], 0.01)
    medians[m] = q[0]
    p95s[m] = q[1]
    print(f"  {m:<35} {q[0]:>8.2f} {q[1]:>8.2f}")

# ---- helper: score 0 at median, full points at p95 ----
def score_above(col_name, max_points):
    m = medians[col_name]
    p = p95s[col_name]
    return when(
        (col(col_name) > m) & (lit(p) > m),
        least(round(((col(col_name) - m) / (lit(p) - m)) * max_points, 1), lit(max_points))
    ).otherwise(0.0)

# ---- apply each component ----
device_scored = device_profile

device_scored = device_scored.withColumn("scr_customers", score_above("distinct_customers", 10))
device_scored = device_scored.withColumn("scr_accounts", score_above("distinct_accounts", 8))
device_scored = device_scored.withColumn("scr_cards", score_above("distinct_cards", 7))

device_scored = device_scored.withColumn("scr_fiu_high", score_above("linked_fiu_high_risk", 15))
device_scored = device_scored.withColumn("scr_fiu_critical", score_above("linked_fiu_critical", 10))

device_scored = device_scored.withColumn("scr_ent_high", score_above("linked_ent_high_risk", 10))
device_scored = device_scored.withColumn("scr_ent_critical", score_above("linked_ent_critical", 5))

device_scored = device_scored.withColumn("scr_merch_high", score_above("linked_high_risk_merchants", 10))
device_scored = device_scored.withColumn("scr_merch_critical", score_above("linked_critical_merchants", 5))

device_scored = device_scored.withColumn("scr_rtc_eft", score_above("rtc_eft_pct", 5))
device_scored = device_scored.withColumn("scr_night", score_above("night_pct", 5))

device_scored = device_scored.withColumn(
    "scr_shared_flag",
    when(col("shared_device_flag") == 1, 5.0).otherwise(0.0)
)
device_scored = device_scored.withColumn(
    "scr_first_txn",
    when(col("has_first_txn") == 1, 5.0).otherwise(0.0)
)

# ---- combine into final score ----
device_scored = device_scored.withColumn(
    "device_risk_score_raw",
    col("scr_customers") + col("scr_accounts") + col("scr_cards") +
    col("scr_fiu_high") + col("scr_fiu_critical") +
    col("scr_ent_high") + col("scr_ent_critical") +
    col("scr_merch_high") + col("scr_merch_critical") +
    col("scr_rtc_eft") + col("scr_night") +
    col("scr_shared_flag") + col("scr_first_txn")
)

device_scored = device_scored.withColumn(
    "device_risk_score",
    round(least(col("device_risk_score_raw"), lit(100.0)), 0).cast("int")
)

# ---- percentile-based risk bands ----
band_cutoffs = device_scored.approxQuantile("device_risk_score", [0.55, 0.80, 0.95], 0.01)

device_scored = device_scored.withColumn(
    "device_risk_band",
    when(col("device_risk_score") >= band_cutoffs[2], "CRITICAL")
    .when(col("device_risk_score") >= band_cutoffs[1], "HIGH")
    .when(col("device_risk_score") >= band_cutoffs[0], "MEDIUM")
    .otherwise("LOW")
)

# ---- reason code ----
device_scored = device_scored.withColumn(
    "top_risk_reason",
    when(col("scr_fiu_critical") >= 8, "CRITICAL_FIU_CUSTOMERS")
    .when(col("scr_fiu_high") >= 12, "HIGH_FIU_CUSTOMERS")
    .when(col("scr_customers") >= 8, "EXCESSIVE_CUSTOMERS")
    .when(col("scr_merch_critical") >= 4, "CRITICAL_MERCHANTS")
    .when(col("scr_shared_flag") >= 5, "SHARED_DEVICE_FLAG")
    .when(col("scr_night") >= 4, "HIGH_NIGHT_ACTIVITY")
    .when(col("scr_rtc_eft") >= 4, "HIGH_RTC_EFT")
    .otherwise("MODERATE_RISK")
)

# ---- show results ----
print("\ndevice risk score statistics:")
device_scored.select("device_risk_score").describe().show()

print("risk band distribution:")
total_devices = device_scored.count()
device_scored.groupBy("device_risk_band").count() \
    .withColumn("pct", round(col("count") / total_devices * 100, 1)) \
    .orderBy(col("count").desc()).show()

population benchmarks for scoring:
  metric                                median      p95
  ----------------------------------- -------- --------
  distinct_customers                      1.00     3.00
  distinct_accounts                       2.00     6.00
  distinct_cards                          2.00     5.00
  linked_fiu_high_risk                    1.00     2.00
  linked_fiu_critical                     0.00     0.00
  linked_ent_high_risk                    0.00     1.00
  linked_ent_critical                     0.00     1.00
  linked_high_risk_merchants              9.00    27.00
  linked_critical_merchants               3.00     9.00
  rtc_eft_pct                             0.28     0.36
  night_pct                               1.00     1.00

device risk score statistics:
+-------+------------------+
|summary| device_risk_score|
+-------+------------------+
|  count|             18854|
|   mean| 24.17205897952689|
| stddev|20.763703341993764|
|    min|                 5|
|  

In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# select final columns and write the mart
# ---------------------------------------------------------------------------

target = f"{gold}.gold_device_linkage_risk_mart"

final = device_scored.select(
    # identity
    "device_id",
    "device_type",
    "device_os",
    "device_manufacturer",
    "device_model",
    "operational_risk_score",
    "device_link_type",
    "is_primary_device",
    "has_first_txn",
    "first_seen",
    "last_seen",
    "active_days",

    # network concentration
    "distinct_customers",
    "distinct_accounts",
    "distinct_cards",
    "distinct_merchants",

    # transaction behaviour
    "total_transactions",
    "total_amount",
    "avg_transaction_amount",
    "max_transaction_amount",
    "avg_daily_transactions",
    "night_pct",
    "weekend_pct",
    "rtc_eft_pct",

    # customer risk propagation
    "linked_fiu_high_risk",
    "linked_fiu_critical",
    "linked_pass_through",
    "linked_rtc_abuse",
    "avg_fiu_score",
    "max_fiu_score",
    "linked_ent_high_risk",
    "linked_ent_critical",
    "avg_ent_score",
    "max_ent_score",

    # merchant risk propagation
    "linked_high_risk_merchants",
    "linked_critical_merchants",
    "avg_merchant_risk_score",
    "max_merchant_risk_score",

    # score components
    "scr_customers",
    "scr_accounts",
    "scr_cards",
    "scr_fiu_high",
    "scr_fiu_critical",
    "scr_ent_high",
    "scr_ent_critical",
    "scr_merch_high",
    "scr_merch_critical",
    "scr_rtc_eft",
    "scr_night",
    "scr_shared_flag",
    "scr_first_txn",

    # final risk
    "device_risk_score",
    "device_risk_band",
    "top_risk_reason",

    # audit
    lit(batch_id).alias("batch_id"),
    current_timestamp().alias("built_at")
)

print(f"writing to {target}")
print(f"rows: {final.count():,}")
print(f"columns: {len(final.columns)}")

final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target)

print("done")

writing to ubuntu_bank_fraud.gold.gold_device_linkage_risk_mart
rows: 18,854
columns: 56
done


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# VALIDATION — structural checks, distribution checks, reality review
# ---------------------------------------------------------------------------

target = f"{gold}.gold_device_linkage_risk_mart"
tbl = spark.table(target)
total = tbl.count()

print("=" * 60)
print("VALIDATION: gold_device_linkage_risk_mart")
print("=" * 60)

# ---- 1. total devices ----
print(f"\n[1] total devices: {total:,}")

# ---- 2. duplicate device ids ----
dupes = tbl.groupBy("device_id").count().filter(col("count") > 1).count()
print(f"[2] duplicate device ids: {dupes} {'OK' if dupes == 0 else 'FAIL'}")

# ---- 3. null device ids ----
nulls = tbl.filter(col("device_id").isNull()).count()
print(f"[3] null device ids: {nulls} {'OK' if nulls == 0 else 'FAIL'}")

# ---- 4. risk band distribution ----
print(f"\n[4] risk band distribution:")
tbl.groupBy("device_risk_band").count() \
    .withColumn("pct", round(col("count") / total * 100, 1)) \
    .orderBy(col("count").desc()).show()

# ---- 5. risk score statistics ----
print(f"\n[5] risk score statistics:")
tbl.select("device_risk_score").describe().show()

# ---- 6. top 20 highest-risk devices ----
print(f"\n[6] top 20 critical devices:")
tbl.filter(col("device_risk_band") == "CRITICAL") \
    .select("device_id", "device_risk_score", "device_risk_band",
            "distinct_customers", "distinct_accounts",
            "linked_fiu_high_risk", "linked_fiu_critical",
            "linked_high_risk_merchants", "linked_critical_merchants") \
    .orderBy(col("device_risk_score").desc()) \
    .show(20, truncate=False)

# ---- 7. averages ----
avg_cust = tbl.select(avg("distinct_customers")).collect()[0][0]
avg_acct = tbl.select(avg("distinct_accounts")).collect()[0][0]
print(f"\n[7] average customers per device: {avg_cust:.2f}")
print(f"    average accounts per device: {avg_acct:.2f}")

# ---- 8. multi-customer and multi-account percentages ----
multi_cust = tbl.filter(col("distinct_customers") > 1).count()
multi_acct = tbl.filter(col("distinct_accounts") > 1).count()
print(f"\n[8] devices linked to multiple customers: {multi_cust:,} ({multi_cust/total*100:.1f}%)")
print(f"    devices linked to multiple accounts:  {multi_acct:,} ({multi_acct/total*100:.1f}%)")

# ---- 9. critical percentage ----
crit = tbl.filter(col("device_risk_band") == "CRITICAL").count()
print(f"\n[9] CRITICAL devices: {crit:,} ({crit/total*100:.1f}%)")

# ---- 10. reality review ----
print(f"\n{'='*60}")
print("REALITY REVIEW")
print("=" * 60)

low_pct = tbl.filter(col("device_risk_band") == "LOW").count() / total * 100
med_pct = tbl.filter(col("device_risk_band") == "MEDIUM").count() / total * 100
high_pct = tbl.filter(col("device_risk_band") == "HIGH").count() / total * 100
crit_pct = tbl.filter(col("device_risk_band") == "CRITICAL").count() / total * 100

print(f"  LOW:      {low_pct:.1f}%")
print(f"  MEDIUM:   {med_pct:.1f}%")
print(f"  HIGH:     {high_pct:.1f}%")
print(f"  CRITICAL: {crit_pct:.1f}%")

checks = []
checks.append(("LOW between 50-70%", 50 <= low_pct <= 70, f"{low_pct:.1f}%"))
checks.append(("CRITICAL between 1-5%", 1 <= crit_pct <= 5, f"{crit_pct:.1f}%"))

all_ok = True
for label, passed, value in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_ok = False
    print(f"  {status}: {label} — got {value}")

if all_ok:
    print(f"\n  ALL CHECKS PASSED — distribution is realistic")
else:
    print(f"\n  SOME CHECKS FAILED — recalibration may be needed")

print(f"\n{'='*60}")
print("VALIDATION COMPLETE")
print("=" * 60)

VALIDATION: gold_device_linkage_risk_mart

[1] total devices: 18,854
[2] duplicate device ids: 0 OK
[3] null device ids: 0 OK

[4] risk band distribution:
+----------------+-----+----+
|device_risk_band|count| pct|
+----------------+-----+----+
|             LOW|10045|53.3|
|          MEDIUM| 4904|26.0|
|            HIGH| 2779|14.7|
|        CRITICAL| 1126| 6.0|
+----------------+-----+----+


[5] risk score statistics:
+-------+------------------+
|summary| device_risk_score|
+-------+------------------+
|  count|             18854|
|   mean| 24.17205897952689|
| stddev|20.763703341993736|
|    min|                 5|
|    max|                84|
+-------+------------------+


[6] top 20 critical devices:
+-----------+-----------------+----------------+------------------+-----------------+--------------------+-------------------+--------------------------+-------------------------+
|device_id  |device_risk_score|device_risk_band|distinct_customers|distinct_accounts|linked_fiu_high_ris